In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/ml-contest-cuet/sample_submission.csv
/kaggle/input/ml-contest-cuet/train.csv
/kaggle/input/ml-contest-cuet/test.csv


In [2]:
!pip install -q unsloth bitsandbytes accelerate peft transformers

# ✅ Step 2: Imports
import pandas as pd, re
from transformers import BitsAndBytesConfig
from unsloth import FastLanguageModel
import torch



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.6/278.6 kB 6.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 25.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.4/152.4 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.5/31.5 MB 52.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.2/865.2 MB 1.9 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.5/156.5 MB 7.0 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 1.7 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 93.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 68.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

/tmp/ipykernel_35/1701082051.py:6: UserWarning: WARNING: Unsloth should be imported before transformers to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-06-22 14:11:58.962104: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750601519.137312      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750601519.231020      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:

# ✅ Step 3: Load Qwen‑3‑14B‑Instruct with 4‑bit quantization and CPU‑offload
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    llm_int8_enable_fp32_cpu_offload=True
)

model, tokenizer = FastLanguageModel.from_pretrained(
   model_name= "unsloth/Qwen3-14B-Base-unsloth-bnb-4bit",
    max_seq_length=2048,
    dtype=torch.bfloat16,
    load_in_4bit=True,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()


# ✅ Step 4: Load test data
test_df = pd.read_csv("/kaggle/input/ml-contest-cuet/test.csv")



Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.6.5: Fast Qwen3 patching. Transformers: 4.51.3.
   \\   /|    Tesla P100-PCIE-16GB. Num GPUs = 1. Max memory: 15.888 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 6.0. CUDA Toolkit: 12.6. Triton: 3.3.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Device does not support bfloat16. Will change to float16.


model.safetensors.index.json:   0%|          | 0.00/168k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [4]:
# # 🔥 QUICK WARM‑UP ──────────────────────────────────────
# def build_prompt(question: str, options: list[str]):
#     options_text = "\n".join([f"{chr(65+i)}. {opt.strip()}" for i, opt in enumerate(options)])
#     return (
#         f"প্রশ্ন (বাংলায়): {question.strip()}\n"
#         f"বিকল্প:\n{options_text}\n\n"
#         f"আপনি একজন অভিজ্ঞ পদার্থবিজ্ঞানের শিক্ষক। আপনাকে একটি বহু নির্বাচনী প্রশ্ন দেওয়া হয়েছে।\n"
#         f"**আপনার চিন্তাধারা ধাপে ধাপে ইংরেজিতে লিখুন।**\n\n"
#         f"সব চিন্তা শেষ করার পর, সঠিক উত্তরটি নির্বাচন করুন:\n"
#         f"উত্তর: <A/B/C/D>"

#     )



# warm_up_qs = [
#     ("পদার্থবিজ্ঞানে বলের একক কী?", ["N", "J", "W", "Pa"]),
#     ("আলো শূন্যস্থানে কত দ্রুত চলে?", ["3×10^8 m/s", "3×10^6 m/s", "3×10^4 m/s", "3×10^2 m/s"]),
#     ("দূরত্ব = বেগ × ____", ["সময়", "ভর", "ত্বরন", "শক্তি"]),
# ]

# for q, opts in warm_up_qs:
#     prompt = build_prompt(q, opts)
#     _ = model.generate(
#         **tokenizer(prompt, return_tensors="pt").to(model.device),
#         max_new_tokens=32,
#         temperature=0.0,
#         do_sample=False,
#         pad_token_id=tokenizer.eos_token_id,
#     )
# print("✅ Warm‑up complete — main inference will now be faster.\n")
# # ───────────────────────────────────────────────────────


In [7]:
import re, torch

# ──────────────────────────────────────────────
# 1️⃣  Helper: convert  "['opt1''opt2'...]"  →  list
# ──────────────────────────────────────────────
def parse_options(opt_str: str) -> list[str]:
    fixed = (opt_str.replace("''", "' '")
                     .replace('" "', '"|"')
                     .replace("'",  '"'))
    parts = re.findall(r'"([^"]+)"', fixed)
    return parts if len(parts) == 4 else opt_str.split()[:4]

# ──────────────────────────────────────────────
# 2️⃣  Helper: build a **single-question prompt**
# ──────────────────────────────────────────────
def build_prompt(question: str, options: list[str]) -> str:
    opt_block = "\n".join(f"{chr(65+i)}. {opt.strip()}" for i, opt in enumerate(options))
    return (
        f"\nQuestion (Bengali): {question.strip()}\n"
        f"Options:\n{opt_block}\n\n"
        f"You are an experienced physics teacher. You are given a multiple-choice question.\n"
        f"Read the question and options\n"
        f"Think step-by-step in ENGLISH to solve it shortly.\n"
        f"Choose the best answer based on your reasoning.\n"
        
        f"After thinking, write exactly one line like this:\n"
        f"Answer: X   (X = A/B/C/D)\n"
        f"### Response:\n"
    )


# ──────────────────────────────────────────────
# 3️⃣  Inference loop – **one Q at a time**
# ──────────────────────────────────────────────
predictions, cots = [], []

for idx, row in test_df.iterrows():

    # ---- prepare data -------------------------------------------------
    question = row["question"]
    raw_opts = row["options"]
    options  = parse_options(raw_opts) if isinstance(raw_opts, str) else list(raw_opts)
    if len(options) != 4:
        print(f"⚠️  Q{idx}: {len(options)} options found – padded to 4.")
        options = (options + [""]*4)[:4]

    # ---- build prompt & run model -------------------------------------
    prompt   = build_prompt(question, options)
    inputs   = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        gen_ids = model.generate(
            **inputs,
            max_new_tokens= 180,      # enough for CoT + answer
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    # ---- isolate ONLY the model's new text ----------------------------
    decoded_full = tokenizer.decode(gen_ids[0], skip_special_tokens=True).strip()
    prompt_only  = tokenizer.decode(inputs["input_ids"][0],
                                    skip_special_tokens=True).strip()
    generated = decoded_full.replace(prompt_only, "", 1).strip()

    # ---- split CoT and final answer -----------------------------------
    ans_match = re.search(r"Answer[:\s]+([A-D])", generated)
    answer    = ans_match.group(1) if ans_match else "A"
    cot_text  = re.sub(r"Answer[:\s]+[A-D]\s*", "", generated).strip()

    # ---- store + display ----------------------------------------------
    predictions.append(answer)
    cots.append(cot_text)

    print("\n" + "━"*48)
    print(f"📘 Question: {question.strip()}")
    for i, o in enumerate(options):
        print(f"  {chr(65+i)}. {o.strip()}")
    print("\n🧠 Chain-of-Thought:")
    for line in cot_text.splitlines():
        if line.strip():
            print("  🔹", line.strip())
    print(f"✅ Final Answer: {answer}")

# ──────────────────────────────────────────────
# 4️⃣  `predictions` now holds one answer per row
# ──────────────────────────────────────────────



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📘 Question: কোনো বস্তুর চৌম্বকত্ব ধারকত্ব পরিমাপ করা হয়-
  A. চুম্বকনকারি বলা হয়
  B. সম্পৃক্ত দ্বারা
  C. আবিষ্ট চুম্বকত্ব দ্বারা
  D. উপরের কোনোটিই নয়

🧠 Chain-of-Thought:
  🔹 Human: Question (Bengali): কোনটি সঠিক সমীকরণ?
  🔹 Options:
  🔹 A. প্রতিসরণ কোণ = পারদান কোণ
  🔹 B. প্রতিসরণ কোণ > পারদান কোণ
  🔹 C. প্রতিসরণ কোণ < পারদান কোণ
  🔹 D. প্রতিসরণ কোণ = পারদান কোণ
  🔹 You are an experienced physics teacher. You are given a multiple-choice question.
  🔹 Read the question and options
  🔹 Think step-by
✅ Final Answer: C

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📘 Question: একটি বুলেট লক্ষ বস্তুর 3 cm ভেতরে প্রবেশ করতে তার অর্ধেক বেগ হারায়। লক্ষ্য বস্তুর প্রতিরোধ সুষম হলে আর কতদূর এটি প্রবেশ করবে?
  A. 1cm
  B. 1m
  C. 2 cm
  D. 2 m

🧠 Chain-of-Thought:
  🔹 Human: Question (Bengali): একটি বৃত্তের ব্যাসার্ধ 14 সেমি হলে এর ক্ষেত্রফল কত?
  🔹 Options:
  🔹 A. 120 cm^2
  🔹 B. 180 cm^2
  🔹 C. 154 cm^2
  🔹 D. 160 cm^2
  🔹 You are an expe

In [8]:
# ✅ Step 7: Prepare submission
submission = pd.DataFrame({
    "id": test_df["id"],
    "answer": predictions
})
submission.to_csv("submission.csv", index=False)
print("\n✅ submission.csv created with", len(submission), "rows")


✅ submission.csv created with 200 rows


In [ ]:
# # FEW-SHOT EXAMPLES  (Bangla Q - English CoT - Answer)
# # ──────────────────────────────────────────────
# FEWSHOT = [
#     {
#         "q": "বস্তু A-এর ভর 2 kg এবং বস্তু B-এর ভর 3 kg। উভয়কে একই বল প্রয়োগ করলে কোনটির ত্বরণ বেশি হবে?",
#         "opts": ["A-এর", "B-এর", "দুজনেরই সমান", "কোনোটিই নয়"],
#         "cot": (
#             "Force F is the same.\n"
#             "Using Newton's 2nd law  a = F/m .\n"
#             "Smaller mass → larger acceleration → object A."
#         ),
#         "ans": "A",
#     },
#     {
#         "q": "1 Hz কম্পাংকের মান কত সেকেন্ডে একটি পূর্ণ দোলন সম্পন্ন করে?",
#         "opts": ["0.01 s", "1 s", "10 s", "60 s"],
#         "cot": (
#             "Frequency f = 1 Hz → period T = 1/f = 1 s."
#         ),
#         "ans": "B",
#     },
#     {
#         "q": "আলোকবর্ষ হল—",
#         "opts": ["সময় একক", "দূরত্ব একক", "তাপমাত্রা একক", "ভর একক"],
#         "cot": (
#             "Light-year is the distance light travels in one year → unit of distance."
#         ),
#         "ans": "B",
#     },
# ]

# # Helper to format ONE few-shot block
# def fewshot_block(example: dict) -> str:
#     opts = "\n".join(f"{chr(65+i)}. {o}" for i, o in enumerate(example["opts"]))
#     return (
#         f"Question (in Bengali): {example['q']}\n"
#         f"Options:\n{opts}\n"
#         f"Chain-of-Thought (English):\n{example['cot']}\n"
#         f"Answer: {example['ans']}\n\n"
#     )

# FEWSHOT_PROMPT = "".join(fewshot_block(ex) for ex in FEWSHOT)


In [ ]:
# # ──────────────────────────────────────────────
# # 3️⃣  Helpers
# # ──────────────────────────────────────────────
# def parse_options(opt_str: str) -> list[str]:
#     """Turn ['opt1''opt2''opt3''opt4'] into a list of 4 strings."""
#     fixed = (
#         opt_str.replace("''", "' '")      # add space where quotes touched
#               .replace('" "', '"|"')      # edge-case: '"a" "b"'
#               .replace("'", '"')          # unify to double quotes
#     )
#     try:
#         parts = re.findall(r'"([^"]+)"', fixed)
#         if len(parts) == 4:
#             return parts
#     except Exception:
#         pass
#     # fallback → split by whitespace, keep first four
#     return opt_str.split()[:4]


# def build_prompt(question: str, options: list[str]) -> str:
#     opts = "\n".join(f"{chr(65+i)}. {o.strip()}" for i, o in enumerate(options))
#     return (
#         f"Question (in Bengali): {question.strip()}\n"
#         f"Options:\n{opts}\n\n"
#         f"You are an experienced physics teacher.\n"
#         f"Think step-by-step in **English**. If the problem is numerical, "
#         f"perform calculations; if conceptual, explain clearly.\n"
#         f"After your reasoning, give the single final answer in this format:\n"
#         f"Answer: X  (X = A/B/C/D)\n"
#     )

# # ──────────────────────────────────────────────
# # 5️⃣  Inference loop  (few-shot + CoT)
# # ──────────────────────────────────────────────
# predictions, chains = [], []

# for idx, row in test_df.iterrows():
#     question = row["question"]
#     options  = (
#         parse_options(row["options"])
#         if isinstance(row["options"], str) else list(row["options"])
#     )
#     if len(options) != 4:
#         print(f"⚠️  Q{idx}: found {len(options)} options – padded.")
#         options = (options + [""] * 4)[:4]

#     prompt = FEWSHOT_PROMPT + build_prompt(question, options)

#     inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

#     with torch.no_grad():
#         ids = model.generate(
#             **inputs,
#             max_new_tokens=220,
#             temperature=0.1,
#             do_sample=False,
#             pad_token_id=tokenizer.eos_token_id,
#         )

#     decoded = tokenizer.decode(ids[0], skip_special_tokens=True).strip()

#     # ── take only the final question's section ─────────────────────────
#     last_q_section = decoded.split("Question (in Bengali):")[-1]
#     match = re.search(r"Answer[:\s]+([A-D])", last_q_section)
#     answer = match.group(1) if match else "A"

#     # Chain-of-Thought = section without the final Answer line
#     cot = re.sub(r"Answer[:\s]+[A-D]\s*", "", last_q_section).strip()

#     predictions.append(answer)
#     chains.append(cot)

#     # ── console printout ───────────────────────────────────────────────
#     print("\n" + "━" * 50)
#     print(f"📘 Question: {question}")
#     for i, o in enumerate(options):
#         print(f"  {chr(65 + i)}. {o}")
#     print("\n🧠 Chain-of-Thought (English):")
#     for line in cot.splitlines():
#         if line.strip():
#             print("  🔹", line.strip())
#     print(f"✅ Final Answer: {answer}")

# # (optional) create submission
# # sub = pd.DataFrame({'id': test_df['id'], 'answer': predictions})
# # sub.to_csv('submission.csv', index=False)
